# Deteksi Jenis Kerusakan Jalan — CNN — Struktur BiLSTM — LOCAL WSL2

CNN murni **dilatih dari awal**, dengan urutan sel, optimizer Adam, training loop `GradientTape`, progress bar, callback manual, dan pesan training mengikuti `Split_Test_Biasa_BiLSTM_LOCAL.ipynb`.

**Dataset tetap RDDC2024-ID, dengan gambar, anotasi, dan anggota train/validation/test yang sama seperti notebook YOLO.** Gunakan kembali `~/deteksi-jalan/data/rddc/` dan `~/deteksi-jalan/split_kelompok.csv`; tidak ada unduhan, cropping, pembuangan gambar, atau pembagian ulang.

Satu gambar bisa memiliki beberapa kelas kerusakan. Karena itu, CNN menggunakan **klasifikasi multi-label**: lima output sigmoid untuk Longitudinal Crack, Lateral Crack, Alligator Crack, Pothole, Other. Contoh label `[1, 0, 0, 1, 0]` berarti ada Longitudinal Crack dan Pothole pada gambar yang sama. Beberapa bbox dengan kelas sama tetap menjadi satu label kehadiran kelas. Label kosong menjadi `[0, 0, 0, 0, 0]`; pastikan file kosong tersebut memang background yang telah diverifikasi. **Other tetap kelas kerusakan kelima, bukan kelas jalan normal.** CNN ini memprediksi jenis pada gambar utuh, tidak menghasilkan bounding box.

**Disamakan dengan BiLSTM:** 15 epoch, batch 64, Adam LR 3e-4 dengan clipnorm 1.0; BinaryCrossentropy; EarlyStopping patience 5/min_delta 1e-4; ReduceLR patience 3/factor 0.5/min 1e-6; `loss / acc / val_loss / val_acc`; TensorBoard; simpan/load best model. `acc` merupakan **binary accuracy per label**, bukan persentase gambar dengan semua label benar. Evaluasi juga menyimpan exact-match accuracy dan F1 macro/micro.

### Cara menjalankan
1. Buka di VS Code **WSL2**, pilih kernel **`.venv-tf` TensorFlow** yang sudah digunakan notebook BiLSTM. CNN ini tidak memakai `.venv-yolo`. Tidak perlu mengganti instalasi TensorFlow yang sudah bekerja dengan GTX 1080 Ti.
2. Periksa `PROJECT_DIR` pada sel konfigurasi. Default `~/deteksi-jalan`, bukan folder dataset teks skripsi.
3. Jalankan semua sel berurutan. Jika dataset jalan belum disiapkan, jalankan Bagian A notebook jalan lama sekali terlebih dahulu.
4. Hasil terpisah di `~/deteksi-jalan/hasil/cnn_bilstm_style/<RUN_NAME>/`. Ubah `RUN_NAME` untuk eksperimen berikutnya. Jika kehabisan VRAM, gunakan batch 32 pada eksperimen baru.

Seperti BiLSTM acuan, notebook ini menjalankan **satu eksperimen per eksekusi**. File checkpoint `.keras` berisi model; tidak mengklaim pemulihan penuh state optimizer/counter setelah kernel berhenti. `last_model_cnn.keras` dapat dipakai sebagai bobot awal fase baru jika diperlukan, dengan nama eksperimen baru.


In [ ]:
# 1 · Import — framework dan gaya yang sama dengan notebook BiLSTM
import os, sys, json, random, hashlib, shutil
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import yaml
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display
from tensorflow.keras import layers, Model, regularizers
from sklearn.metrics import (
    classification_report, accuracy_score, hamming_loss,
    precision_recall_fscore_support, multilabel_confusion_matrix,
)
SEED = 42
tf.keras.utils.set_random_seed(SEED)
# Jika import pendukung belum tersedia, instal HANYA paket yang hilang di .venv-tf:
# python -m pip install pandas scikit-learn matplotlib pillow pyyaml tqdm ipywidgets
# Jangan memasang PyTorch/CUDA YOLO di environment ini.


In [ ]:
# 2 · Verifikasi kernel TensorFlow WSL2 dan GTX 1080 Ti — sama seperti BiLSTM
print('Python:', sys.executable)
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU yang terdeteksi:', gpus)
if not gpus:
    raise RuntimeError('GPU tidak terdeteksi oleh kernel notebook. '
                       'Pilih kernel ~/skripsi-training/.venv-tf/bin/python melalui WSL2, '
                       'seperti saat menjalankan notebook BiLSTM.')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        print('GPU sudah diinisialisasi; melanjutkan tanpa mengubah memory_growth.')
# Tidak cukup hanya terdeteksi: pastikan operasi CNN dapat dijalankan pada GPU.
with tf.device('/GPU:0'):
    probe = tf.nn.conv2d(tf.ones([1, 16, 16, 3]), tf.ones([3, 3, 3, 4]), strides=1, padding='SAME')
    tf.debugging.assert_all_finite(probe, 'Uji CUDA menghasilkan NaN/Inf')
    _ = probe.numpy()
    if 'GPU' not in probe.device.upper():
        raise RuntimeError(f'Operasi CNN tidak berjalan pada GPU: {probe.device}')
print('Uji komputasi CNN di GPU: OK')
del probe


In [ ]:
# 3 · Konfigurasi project dan dataset jalan yang sama seperti notebook YOLO
PROJECT_DIR = Path(os.environ.get('PROJ_DIR', str(Path.home() / 'deteksi-jalan'))).expanduser().resolve()
DATASET_DIR = PROJECT_DIR / 'data' / 'rddc'
GROUP_CSV = PROJECT_DIR / 'split_kelompok.csv'
SPLIT = 'kelompok'     # tetap split lama; acak hanya jika memakai data_random.yaml lama
RUN_NAME = 'cnn_from_scratch_01'
IMG_SIZE = 224
THRESHOLD = 0.5       # keputusan per kelas: probabilitas > 0.5, sama dengan BinaryAccuracy Keras
CLASS_NAMES = ['Longitudinal Crack', 'Lateral Crack', 'Alligator Crack', 'Pothole', 'Other']
NUM_CLASSES = len(CLASS_NAMES)
if SPLIT not in {'kelompok', 'acak'}:
    raise ValueError('SPLIT harus kelompok atau acak.')
SOURCE_YAML = DATASET_DIR / ('data.yaml' if SPLIT == 'kelompok' else 'data_random.yaml')
SPLIT_DIR = PROJECT_DIR / 'hasil' / 'cnn_bilstm_style' / RUN_NAME
MODELS_DIR = SPLIT_DIR / 'models'
print('Folder project:', PROJECT_DIR)
print('Dataset:', SOURCE_YAML)
print('Folder output lokal:', SPLIT_DIR)


In [ ]:
# 4 · Muat dataset lokal yang SAMA (tanpa unduh atau membuat split ulang)
if not SOURCE_YAML.is_file():
    raise FileNotFoundError(f'Dataset belum ditemukan: {SOURCE_YAML}\n'
                            'Periksa PROJECT_DIR atau siapkan dataset sekali menggunakan '
                            'Bagian A notebook kerusakan jalan lama.')
source_config = yaml.safe_load(SOURCE_YAML.read_text(encoding='utf-8'))
raw_names = source_config.get('names', {})
if isinstance(raw_names, dict) and sorted(map(int, raw_names)) != list(range(len(CLASS_NAMES))):
    raise ValueError('ID kelas YAML harus 0 sampai 4 dengan urutan yang sama seperti dataset lama.')
names = ([raw_names[k] for k in sorted(raw_names, key=int)]
         if isinstance(raw_names, dict) else list(raw_names))
if names != CLASS_NAMES:
    raise ValueError(f'Urutan/nama kelas berbeda dari notebook jalan lama: {names}')

# YAML lama menunjuk daftar train/val/test.txt. Anggota daftar dipertahankan.
def resolve_image(raw, list_path):
    p = Path(raw).expanduser()
    candidates = [p] if p.is_absolute() else [list_path.parent / p, DATASET_DIR / p]
    # Dukungan memindahkan project WSL: hanya relokasi file images dengan nama sama.
    candidates.append(DATASET_DIR / 'images' / p.name)
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(f'Gambar dari {list_path.name} tidak ditemukan: {raw}')

rows = []
for part in ('train', 'val', 'test'):
    entry = source_config.get(part)
    if not isinstance(entry, str) or Path(entry).suffix.lower() != '.txt':
        raise ValueError(f'{part} harus menunjuk file .txt split dari notebook lama.')
    list_path = Path(entry).expanduser()
    if not list_path.is_absolute():
        list_path = DATASET_DIR / list_path
    if not list_path.is_file():
        raise FileNotFoundError(f'Daftar split belum ditemukan: {list_path}')
    members = [line.strip() for line in list_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not members:
        raise ValueError(f'Split {part} kosong.')
    for raw in members:
        image = resolve_image(raw, list_path)
        label = DATASET_DIR / 'labels' / (image.stem + '.txt')
        if image.parent != (DATASET_DIR / 'images').resolve():
            raise ValueError(f'Gambar harus memakai images/ dari dataset lama: {image}')
        rows.append(dict(stem=image.stem, image=str(image), label=str(label.resolve()), split=part))
df = pd.DataFrame(rows)
print('Jumlah awal:', len(df))
print(df['split'].value_counts().reindex(['train', 'val', 'test']))
print('Shape:', df.shape)


In [ ]:
# 5 · Audit label dan file (READ-ONLY; dataset sama seperti YOLO)
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

issues, image_hashes, label_hashes, n_boxes = [], [], [], []
for row in tqdm(df.itertuples(), total=len(df), desc='Audit dataset', unit='foto'):
    image_hashes.append(sha256_file(row.image))
    try:
        with Image.open(row.image) as im:
            im.verify()
    except Exception as error:
        issues.append((row.stem, 'gambar rusak', str(error)))
    label = Path(row.label)
    if not label.is_file():
        issues.append((row.stem, 'label hilang', str(label)))
        label_hashes.append('MISSING')
        n_boxes.append(0)
        continue
    label_hashes.append(sha256_file(label))
    lines = [line for line in label.read_text(encoding='utf-8').splitlines() if line.strip()]
    n_boxes.append(len(lines))
    for line_number, line in enumerate(lines, 1):
        try:
            values = list(map(float, line.split()))
            if len(values) != 5 or not np.isfinite(values).all():
                raise ValueError('harus lima angka finite')
            cls, x, y, w, h = values
            if not cls.is_integer() or not 0 <= cls < len(CLASS_NAMES):
                raise ValueError('ID kelas tidak valid')
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                raise ValueError('koordinat YOLO tidak valid')
        except ValueError as error:
            issues.append((row.stem, f'label baris {line_number}', str(error)))
df['sha256_image'], df['sha256_label'], df['n_boxes'] = image_hashes, label_hashes, n_boxes
print('Jumlah file/baris bermasalah:', len(issues))
print('Foto tanpa kotak (background):', int((df.n_boxes == 0).sum()))
if issues:
    display(pd.DataFrame(issues, columns=['stem', 'bagian', 'masalah']))
    raise RuntimeError('Audit belum lolos. Periksa anotasi sumber; notebook ini tidak mengeditnya.')
print('Jumlah setelah audit:', len(df))


In [ ]:
# 6 · Konversi label YOLO menjadi multi-hot pada GAMBAR UTUH, tanpa mengganti dataset
# Tidak memakai argmax/kelas mayoritas agar kelas kerusakan lain tidak hilang.
def yolo_to_multihot(label_path, num_classes=5):
    target = np.zeros(num_classes, dtype=np.float32)
    for line_number, line in enumerate(Path(label_path).read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        values = list(map(float, line.split()))
        if len(values) != 5 or not np.isfinite(values).all():
            raise ValueError(f'Anotasi tidak valid: {label_path}, baris {line_number}')
        class_id = values[0]
        if not class_id.is_integer() or not 0 <= class_id < num_classes:
            raise ValueError(f'ID kelas tidak valid: {label_path}, baris {line_number}')
        target[int(class_id)] = 1.0
    return target

targets = np.stack([yolo_to_multihot(p, NUM_CLASSES) for p in df['label']])
for class_id in range(NUM_CLASSES):
    df[f'target_{class_id}'] = targets[:, class_id].astype(int)
print('Jumlah gambar:', len(df))
print('Jumlah gambar multi-label:', int((targets.sum(axis=1) > 1).sum()))
print('Jumlah gambar beranotasi kosong:', int((targets.sum(axis=1) == 0).sum()))
print('Label kosong dipertahankan; verifikasi bahwa itu benar-benar background, bukan label hilang.')
print('Shape target:', targets.shape)
display(df[['stem', 'split'] + [f'target_{i}' for i in range(NUM_CLASSES)]].head())


In [ ]:
# 7 · Validasi overlap seperti notebook BiLSTM; anggota split jalan tidak diubah
if df['image'].duplicated().any() or df['stem'].duplicated().any():
    raise RuntimeError('Ada gambar/stem terdaftar lebih dari satu kali pada daftar split.')
checks = {part: set(df.loc[df.split == part, 'sha256_image']) for part in ('train', 'val', 'test')}
overlaps = {
    'Jumlah overlap': len(checks['train'] & checks['test']),
    'Overlap Train-Validation': len(checks['train'] & checks['val']),
    'Overlap Validation-Test': len(checks['val'] & checks['test']),
}
for message, count in overlaps.items():
    print(f'{message}: {count}')
if SPLIT == 'kelompok' and any(overlaps.values()):
    raise RuntimeError('Ada isi gambar identik di lebih dari satu split. Gunakan split kelompok lama yang valid.')
if SPLIT == 'kelompok':
    if not GROUP_CSV.is_file():
        raise FileNotFoundError(f'Manifest kelompok lama belum ditemukan: {GROUP_CSV}')
    groups = pd.read_csv(GROUP_CSV, dtype={'stem': str})
    if not {'stem', 'group', 'split'}.issubset(groups.columns) or groups['stem'].duplicated().any():
        raise ValueError('Manifest kelompok lama harus memiliki stem unik, group, dan split.')
    df = df.merge(groups[['stem', 'group', 'split']].rename(columns={'split': 'split_lama'}),
                  on='stem', how='left', validate='one_to_one')
    if len(groups) != len(df) or df['group'].isna().any() or not df['split'].eq(df['split_lama']).all():
        raise RuntimeError('Daftar split tidak cocok dengan split_kelompok.csv lama.')
    if df.groupby('group')['split'].nunique().max() > 1:
        raise RuntimeError('Ada kelompok foto hampir kembar terbelah antar split.')
    print('Overlap kelompok antar split: 0')
else:
    print('PERINGATAN: split acak lama bisa memuat foto hampir kembar lintas split; hanya untuk pembanding.')
print('Split lama dipertahankan; tidak ada data yang dibagi ulang.')


In [ ]:
# 8 · Gunakan anggota train/validation/test lama; tidak memakai train_test_split lagi
X_train = df.loc[df.split == 'train', 'image'].to_numpy(dtype=str)
X_val = df.loc[df.split == 'val', 'image'].to_numpy(dtype=str)
X_test = df.loc[df.split == 'test', 'image'].to_numpy(dtype=str)
target_columns = [f'target_{i}' for i in range(NUM_CLASSES)]
y_train = df.loc[df.split == 'train', target_columns].to_numpy(dtype=np.float32)
y_val = df.loc[df.split == 'val', target_columns].to_numpy(dtype=np.float32)
y_test = df.loc[df.split == 'test', target_columns].to_numpy(dtype=np.float32)
idx_test = df.index[df.split == 'test'].to_numpy()
print(f'Training: {len(X_train)}')
print(f'Validation: {len(X_val)}')
print(f'Testing: {len(X_test)}')
class_distribution = pd.DataFrame({part: labels.sum(axis=0).astype(int)
                                   for part, labels in [('train', y_train), ('val', y_val), ('test', y_test)]},
                                  index=CLASS_NAMES)
class_distribution.loc['Jumlah foto'] = [len(X_train), len(X_val), len(X_test)]
display(class_distribution)
print('Tabel menghitung gambar yang memuat kelas, bukan jumlah bounding box.')


In [ ]:
# 9 · Simpan konfigurasi label dan identitas split (padanan penyimpanan tokenizer)
if SPLIT_DIR.exists() and any(SPLIT_DIR.iterdir()):
    raise FileExistsError(f'Folder eksperimen sudah berisi hasil: {SPLIT_DIR}\n'
                          'Gunakan RUN_NAME baru untuk training baru agar hasil lama tidak tertimpa.')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
fingerprint_rows = df[['stem', 'split', 'sha256_image', 'sha256_label']].sort_values(['split', 'stem'])
DATASET_FINGERPRINT = hashlib.sha256(fingerprint_rows.to_csv(index=False).encode()).hexdigest()
# Algoritme fingerprint identik dengan notebook YOLO struktur BiLSTM.
preprocess_config = dict(img_size=IMG_SIZE, channels=3, resize='resize_with_pad',
                         input_range=[0, 255], rescaling_in_model='1/255',
                         class_names=CLASS_NAMES, threshold=THRESHOLD,
                         decision_rule='probability > threshold', task='multilabel_classification',
                         label_source='presence of each YOLO class in whole image',
                         dataset_fingerprint=DATASET_FINGERPRINT)
preprocess_path = SPLIT_DIR / 'preprocessing.json'
preprocess_path.write_text(json.dumps(preprocess_config, indent=2), encoding='utf-8')
df.to_csv(SPLIT_DIR / 'split_manifest.csv', index=False)
class_distribution.to_csv(SPLIT_DIR / 'class_distribution.csv', index_label='kelas')
print('Preprocessing config saved!')
print('File ada?', preprocess_path.is_file())
print('Ukuran file:', preprocess_path.stat().st_size)
print('Dataset fingerprint:', DATASET_FINGERPRINT)


In [ ]:
# 10 · Load & resize gambar (padanan Tokenisasi & Padding pada BiLSTM)
# Gambar masih 0..255. Rescaling 1/255 ada DI DALAM model agar tersimpan pada .keras.
# Tidak memuat semua gambar ke RAM; data dibaca per batch.
def load_image(path, target):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE, method='bilinear')
    image = tf.cast(image, tf.float32)
    image = tf.ensure_shape(image, [IMG_SIZE, IMG_SIZE, 3])
    return image, tf.cast(target, tf.float32)

print(f'X_train shape (setelah resize): ({len(X_train)}, {IMG_SIZE}, {IMG_SIZE}, 3)')
print(f'X_test shape (setelah resize): ({len(X_test)}, {IMG_SIZE}, {IMG_SIZE}, 3)')
print(f'X_val shape (setelah resize): ({len(X_val)}, {IMG_SIZE}, {IMG_SIZE}, 3)')
print('y_train shape:', y_train.shape)


In [ ]:
# 11 · Bangun Model CNN dari awal — urutan Input → CNN → Dense → Output
# Tidak menggunakan model pretrained, YOLO, BiLSTM, atau transfer learning.
L2 = 1e-4
USE_AUGMENTATION = True
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_layer')
x = inputs
if USE_AUGMENTATION:
    # Augmentasi otomatis aktif hanya ketika training=True.
    # Tanpa rotasi 90 derajat agar longitudinal/lateral tidak tertukar.
    x = layers.RandomFlip('horizontal', seed=SEED, name='flip_horizontal')(x)
    x = layers.RandomContrast(0.15, seed=SEED + 1, name='contrast')(x)
x = layers.Rescaling(1.0 / 255.0, name='rescaling')(x)
for block, filters in enumerate((32, 64, 128, 256), 1):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False,
                      kernel_initializer='glorot_uniform',
                      kernel_regularizer=regularizers.l2(L2), name=f'conv_{block}')(x)
    x = layers.BatchNormalization(name=f'batchnorm_{block}')(x)
    x = layers.Activation('relu', name=f'relu_{block}')(x)
    x = layers.MaxPooling2D(2, name=f'maxpool_{block}')(x)
x = layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
x = layers.Dense(48, activation='relu', kernel_regularizer=regularizers.l2(L2), name='dense')(x)
x = layers.Dropout(0.4, seed=SEED + 2, name='dropout')(x)
outputs = layers.Dense(NUM_CLASSES, activation='sigmoid', name='output')(x)
model = Model(inputs, outputs, name='CNN_RoadDamage_FromScratch')
model.summary()
print('Inisialisasi: bobot acak (from scratch)')
print('Output:', NUM_CLASSES, 'kelas sigmoid multi-label')


In [ ]:
# 12 · Distribusi label training (informasi keseimbangan kelas)
train_positive = y_train.sum(axis=0).astype(int)
train_negative = len(y_train) - train_positive
display(pd.DataFrame({'kelas': CLASS_NAMES, 'positive': train_positive, 'negative': train_negative}))
# BiLSTM acuan menghitung class_weight_dict, tetapi training loop-nya tidak menggunakannya.
# Di sini BCE juga tidak diberi class weights agar objective tidak diam-diam berubah.
class_weight_dict = None
print(f'Class weights: {class_weight_dict}')
if np.any(train_positive == 0):
    print('Kelas tanpa contoh positif di TRAIN:', [CLASS_NAMES[i] for i in np.flatnonzero(train_positive == 0)])


In [ ]:
# 13 · Custom Callback — nilai optimizer dan kontrol SAMA dengan BiLSTM acuan
EPOCHS        = 15
BATCH_SIZE    = 64      # sama dengan BiLSTM; bila OOM gunakan 32 pada RUN_NAME baru
LEARNING_RATE = 3e-4
ES_PATIENCE   = 5
ES_MIN_DELTA  = 1e-4
LR_PATIENCE   = 3
LR_FACTOR     = 0.5
LR_MIN        = 1e-6
CLIP_NORM     = 1.0

best_val_loss       = np.inf
best_epoch          = 0
patience_counter    = 0
lr_patience_counter = 0
current_lr          = LEARNING_RATE
optimizer = tf.keras.optimizers.Adam(learning_rate=current_lr, clipnorm=CLIP_NORM)
loss_fn = tf.keras.losses.BinaryCrossentropy()
# Tidak ada scheduler/optimizer kedua atau weight_decay tambahan.
log_dir = SPLIT_DIR / 'logs'
log_dir.mkdir(exist_ok=True)
writer = tf.summary.create_file_writer(str(log_dir))
checkpoint_path = MODELS_DIR / 'best_model_cnn.keras'
last_checkpoint_path = MODELS_DIR / 'last_model_cnn.keras'
log_path = SPLIT_DIR / 'log_per_epoch.txt'

def log_print(message):
    tqdm.write(str(message))
    with log_path.open('a', encoding='utf-8') as f:
        f.write(str(message) + '\n')

config = dict(model='CNN_from_scratch', task='multilabel_classification', seed=SEED,
              tensorflow=tf.__version__, run_name=RUN_NAME, split=SPLIT,
              dataset_fingerprint=DATASET_FINGERPRINT, optimizer='Adam',
              epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
              clipnorm=CLIP_NORM, es_patience=ES_PATIENCE, es_min_delta=ES_MIN_DELTA,
              lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR, lr_min=LR_MIN,
              loss='BinaryCrossentropy + explicit model L2 regularization',
              l2=L2, use_augmentation=USE_AUGMENTATION, threshold=THRESHOLD,
              class_weights=None, pretrained=False, img_size=IMG_SIZE, classes=CLASS_NAMES)
(SPLIT_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
log_print(f'Optimizer: Adam | LR: {LEARNING_RATE:.6f} | Clip norm: {CLIP_NORM}')
log_print(f'EPOCHS: {EPOCHS} | BATCH_SIZE: {BATCH_SIZE} | Dataset: RDDC2024-ID')
log_print('acc / val_acc = binary accuracy per label; exact-match accuracy dilaporkan saat evaluasi.')


In [ ]:
# 14 · Semua Callback Manual — pesan sama seperti notebook BiLSTM
# Fungsi dipanggil sekali per epoch, setelah training dan validation selesai.
def update_callbacks(epoch, avg_val_loss):
    global best_val_loss, best_epoch, patience_counter, lr_patience_counter, current_lr
    if not np.isfinite(avg_val_loss):
        raise RuntimeError('val_loss NaN/Inf; checkpoint terbaik tidak boleh dipilih dari loss invalid.')
    if avg_val_loss < best_val_loss - ES_MIN_DELTA:
        log_print(f'💾 val_loss improved ({best_val_loss:.4f} → {avg_val_loss:.4f}), saving model...')
        model.save(checkpoint_path)
        best_val_loss = avg_val_loss
        best_epoch = epoch
        patience_counter = 0
        lr_patience_counter = 0
    else:
        patience_counter += 1
        lr_patience_counter += 1
        log_print(f' No improvement. EarlyStopping: {patience_counter}/{ES_PATIENCE} | ReduceLR: {lr_patience_counter}/{LR_PATIENCE}')
    if lr_patience_counter >= LR_PATIENCE:
        new_lr = max(current_lr * LR_FACTOR, LR_MIN)
        if new_lr < current_lr:
            current_lr = new_lr
            optimizer.learning_rate.assign(current_lr)
            log_print(f' ReduceLROnPlateau: LR diturunkan → {current_lr:.6f}')
        lr_patience_counter = 0
    if patience_counter >= ES_PATIENCE:
        log_print(f'\n EarlyStopping triggered! Best val_loss: {best_val_loss:.4f}')
        return True
    return False


In [ ]:
# 15 · Dataset batch (padanan tf.data pada BiLSTM)
# Test tidak di-shuffle. Validation/test tidak diberi augmentasi.
def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(min(10000, len(paths)), seed=SEED, reshuffle_each_iteration=True)
    return (dataset.map(load_image, num_parallel_calls=2)
            .batch(BATCH_SIZE).prefetch(1))

train_dataset = make_dataset(X_train, y_train, training=True)
val_dataset = make_dataset(X_val, y_val)
test_dataset = make_dataset(X_test, y_test)
train_acc_metric = tf.keras.metrics.BinaryAccuracy(threshold=THRESHOLD)
val_acc_metric = tf.keras.metrics.BinaryAccuracy(threshold=THRESHOLD)
train_loss_metric = tf.keras.metrics.Mean()
val_loss_metric = tf.keras.metrics.Mean()
print('Data pipeline siap. Augmentasi train ada di dalam model.')


In [ ]:
# 16 · Training Loop GradientTape + Semua Callback Manual — seperti BiLSTM
@tf.function(reduce_retracing=True)
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        preds = model(x_batch, training=True)
        data_loss = loss_fn(y_batch, preds)
        # L2 layer baru berpengaruh jika model.losses dimasukkan dalam manual loop.
        reg_loss = tf.add_n(model.losses) if model.losses else tf.constant(0.0, dtype=data_loss.dtype)
        loss = data_loss + reg_loss
        tf.debugging.assert_all_finite(loss, 'Training loss NaN/Inf')
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients([(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None])
    return loss, preds

@tf.function(reduce_retracing=True)
def val_step(x_batch, y_batch):
    preds = model(x_batch, training=False)
    data_loss = loss_fn(y_batch, preds)
    reg_loss = tf.add_n(model.losses) if model.losses else tf.constant(0.0, dtype=data_loss.dtype)
    loss = data_loss + reg_loss
    tf.debugging.assert_all_finite(loss, 'Validation loss NaN/Inf')
    return loss, preds

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
learning_rates = []
try:
    for epoch in range(EPOCHS):
        log_print(f'\n Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.6f}')
        learning_rates.append(current_lr)
        train_acc_metric.reset_state()
        val_acc_metric.reset_state()
        train_loss_metric.reset_state()
        val_loss_metric.reset_state()
        with tqdm(train_dataset, desc='  Training', unit='batch', leave=False) as train_bar:
            for x_batch, y_batch in train_bar:
                loss, preds = train_step(x_batch, y_batch)
                train_acc_metric.update_state(y_batch, preds)
                # Rata-rata loss dibobot jumlah gambar, termasuk batch terakhir yang lebih kecil.
                train_loss_metric.update_state(loss, sample_weight=tf.cast(tf.shape(y_batch)[0], tf.float32))
                train_bar.set_postfix({'loss': f'{float(loss):.4f}',
                                       'acc': f'{float(train_acc_metric.result()):.4f}'})
        with tqdm(val_dataset, desc='  Validasi ', unit='batch', leave=False) as val_bar:
            for x_batch, y_batch in val_bar:
                val_loss, val_preds = val_step(x_batch, y_batch)
                val_acc_metric.update_state(y_batch, val_preds)
                val_loss_metric.update_state(val_loss, sample_weight=tf.cast(tf.shape(y_batch)[0], tf.float32))
                val_bar.set_postfix({'val_loss': f'{float(val_loss):.4f}',
                                     'val_acc': f'{float(val_acc_metric.result()):.4f}'})
        avg_loss = float(train_loss_metric.result())
        avg_val_loss = float(val_loss_metric.result())
        avg_acc = float(train_acc_metric.result())
        avg_val_acc = float(val_acc_metric.result())
        history['loss'].append(avg_loss)
        history['accuracy'].append(avg_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_accuracy'].append(avg_val_acc)
        log_print(f' loss: {avg_loss:.4f} | acc: {avg_acc:.4f} | val_loss: {avg_val_loss:.4f} | val_acc: {avg_val_acc:.4f}')
        with writer.as_default():
            tf.summary.scalar('loss', avg_loss, step=epoch)
            tf.summary.scalar('accuracy', avg_acc, step=epoch)
            tf.summary.scalar('val_loss', avg_val_loss, step=epoch)
            tf.summary.scalar('val_accuracy', avg_val_acc, step=epoch)
            tf.summary.scalar('learning_rate', current_lr, step=epoch)
        writer.flush()
        should_stop = update_callbacks(epoch+1, avg_val_loss)
        model.save(last_checkpoint_path)
        history_table = pd.DataFrame(history)
        history_table.insert(0, 'epoch', np.arange(1, len(history_table)+1))
        history_table['learning_rate'] = learning_rates
        history_table.to_csv(SPLIT_DIR / 'history.csv', index=False)
        if should_stop:
            break
    log_print('\n Training selesai!')
except KeyboardInterrupt:
    log_print('\n Training terputus. Checkpoint best/last yang sudah lengkap tetap ada. '
              'Menjalankan loop ini dari awal bukan resume optimizer/counter.')
    raise
finally:
    writer.close()

# Restore best model seperti notebook BiLSTM (tidak memerlukan custom layer).
model = tf.keras.models.load_model(checkpoint_path, compile=False)
log_print(' Best model loaded!')
log_print(f' Best epoch: {best_epoch} | Best val_loss: {best_val_loss:.4f}')


In [ ]:
# 17 · Plot — struktur tiga panel sama dengan BiLSTM
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epoch_axis = np.arange(1, len(history['loss']) + 1)
axes[0].plot(epoch_axis, history['loss'], label='Train Loss')
axes[0].plot(epoch_axis, history['val_loss'], label='Val Loss')
axes[0].set_title('Loss - CNN')
axes[1].plot(epoch_axis, history['accuracy'], label='Train Accuracy')
axes[1].plot(epoch_axis, history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Binary Label Accuracy - CNN')
gap_series = np.asarray(history['accuracy']) - np.asarray(history['val_accuracy'])
axes[2].plot(epoch_axis, gap_series, color='red', label='Train - Val Accuracy Gap')
axes[2].axhline(y=0.05, color='orange', linestyle='--', label='Panduan visual 5%')
axes[2].set_title('Accuracy Gap (diagnostik)')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
fig.savefig(SPLIT_DIR / 'kurva_training.png', dpi=160, bbox_inches='tight')
plt.show()
print('Garis gap 5% hanya panduan visual, bukan kesimpulan otomatis overfitting.')


In [ ]:
# 18 · Evaluasi TEST — hanya memakai best model yang dipilih dari validation
probabilities = model.predict(test_dataset, verbose=0)
y_pred = (probabilities > THRESHOLD).astype(np.int32)
if probabilities.shape != y_test.shape:
    raise RuntimeError('Bentuk prediksi dan label test tidak cocok.')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
label_accuracy = float(np.mean(y_pred == y_test))
exact_match_accuracy = float(accuracy_score(y_test, y_pred))
metrics = pd.DataFrame([dict(
    model='CNN_from_scratch', split=SPLIT, threshold=THRESHOLD,
    jumlah_training=len(X_train), jumlah_validation=len(X_val), jumlah_testing=len(X_test),
    best_epoch=best_epoch, accuracy=label_accuracy, exact_match_accuracy=exact_match_accuracy,
    precision_macro=report['macro avg']['precision'], recall_macro=report['macro avg']['recall'],
    f1_macro=report['macro avg']['f1-score'], f1_micro=report['micro avg']['f1-score'],
    f1_weighted=report['weighted avg']['f1-score'], hamming_loss=float(hamming_loss(y_test, y_pred)),
    dataset_fingerprint=DATASET_FINGERPRINT,
)])
print(metrics.to_string(index=False, float_format=lambda v: f'{v:.5f}'))
print('accuracy = keputusan per label yang benar; exact_match_accuracy = seluruh label gambar harus benar.')
print('Task klasifikasi gambar: tidak memakai mAP@IoU deteksi bounding box.')


In [ ]:
# 19 · Simpan metrics, hasil per kelas, dan prediksi — seperti notebook BiLSTM
metrics.to_csv(SPLIT_DIR / 'metrics.csv', index=False)
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, zero_division=0)
per_class = pd.DataFrame({'class_id': np.arange(NUM_CLASSES), 'kelas': CLASS_NAMES,
                          'precision': precision, 'recall': recall, 'f1': f1, 'support': support})
per_class.to_csv(SPLIT_DIR / 'metrics_per_class.csv', index=False)
prediction_table = pd.DataFrame({'test_index': idx_test, 'image': X_test,
                                 'y_true': [json.dumps([CLASS_NAMES[i] for i in np.flatnonzero(row)]) for row in y_test],
                                 'y_pred': [json.dumps([CLASS_NAMES[i] for i in np.flatnonzero(row)]) for row in y_pred]})
for i in range(NUM_CLASSES):
    prediction_table[f'y_true_{i}'] = y_test[:, i].astype(int)
    prediction_table[f'y_pred_{i}'] = y_pred[:, i]
    prediction_table[f'prob_{i}'] = probabilities[:, i]
prediction_table.to_csv(SPLIT_DIR / 'predictions_test.csv', index=False)
confusion = multilabel_confusion_matrix(y_test, y_pred)
confusion_table = []
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, ax in enumerate(axes.ravel()):
    if i >= NUM_CLASSES:
        ax.axis('off')
        continue
    matrix = confusion[i]
    ax.imshow(matrix, cmap='Blues')
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha='center', va='center',
                color='white' if value > matrix.max() / 2 else 'black')
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Tidak', 'Ada'],
           yticklabels=['Tidak', 'Ada'], xlabel='Prediksi', ylabel='Aktual', title=CLASS_NAMES[i])
    tn, fp, fn, tp = matrix.ravel()
    confusion_table.append(dict(kelas=CLASS_NAMES[i], TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp)))
plt.tight_layout()
fig.savefig(SPLIT_DIR / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()
pd.DataFrame(confusion_table).to_csv(SPLIT_DIR / 'confusion_matrix.csv', index=False)
print('Hasil evaluasi tersimpan di:', SPLIT_DIR)


In [ ]:
# 20 · Simpan Model — model sekarang sudah restore best weights
model_path = MODELS_DIR / 'model_cnn.keras'
model.save(model_path)
model_json = model.to_json()
(SPLIT_DIR / 'model_architecture.json').write_text(model_json, encoding='utf-8')
with (SPLIT_DIR / 'model_summary.txt').open('w', encoding='utf-8') as summary_file:
    model.summary(print_fn=lambda text, **kwargs: summary_file.write(text + '\n'))
print('Model saved!')
print('Model utama:', model_path)
print('Best model:', checkpoint_path)
print('Model terakhir:', last_checkpoint_path)


In [ ]:
# 21 · Inference — Label / Confidence / Raw Score seperti notebook BiLSTM
inference_model = tf.keras.models.load_model(MODELS_DIR / 'model_cnn.keras', compile=False)
def predict_road(image_path, model, threshold=THRESHOLD):
    image, _ = load_image(tf.constant(str(image_path)), tf.zeros(NUM_CLASSES, dtype=tf.float32))
    scores = model(tf.expand_dims(image, axis=0), training=False).numpy()[0]
    active = np.flatnonzero(scores > threshold)
    print(f'Gambar    : {image_path}')
    if len(active):
        for i in active:
            print(f'Label     : {CLASS_NAMES[i]}')
            print(f'Confidence: {scores[i]:.2%}')
            print(f'Raw Score : {scores[i]:.4f}')
    else:
        print('Label     : Tidak ada kelas kerusakan melewati threshold')
    print('Skor semua kelas:')
    for name, score in zip(CLASS_NAMES, scores):
        print(f'  {name}: {score:.4f}')
    with Image.open(image_path) as im:
        plt.figure(figsize=(10, 6))
        plt.imshow(im.convert('RGB'))
        plt.axis('off')
        plt.title(', '.join(CLASS_NAMES[i] for i in active) if len(active) else 'Tidak melewati threshold')
        plt.show()
    return {name: float(score) for name, score in zip(CLASS_NAMES, scores)}

CONTOH_GAMBAR = X_test[0]  # ganti dengan path gambar jalan milikmu
print('       HASIL INFERENCE MODEL CNN')
contoh_hasil = predict_road(CONTOH_GAMBAR, inference_model)


In [ ]:
# 22 · TensorBoard — sama seperti notebook BiLSTM
import shlex
print('Di terminal WSL2 jalankan:')
print(f'tensorboard --logdir {shlex.quote(str(log_dir))} --port 6006')
print('Lalu buka http://localhost:6006 di browser Windows.')
print('Log per epoch:', log_path)


### Kesamaan dan penyesuaian

| Bagian | Implementasi CNN |
|---|---|
| Framework, GPU setup | TensorFlow / Keras dan memory growth, seperti BiLSTM |
| Optimizer | Adam, learning_rate=3e-4, clipnorm=1.0, tanpa weight decay tambahan |
| Epoch / batch | 15 / 64, sama dengan BiLSTM |
| Objective | BinaryCrossentropy per kelas; L2 layer dimasukkan eksplisit pada loop manual |
| Progress bar | Training: loss/acc; Validasi: val_loss/val_acc |
| Ringkasan epoch | `Epoch n/N | LR`, lalu `loss | acc | val_loss | val_acc` |
| Checkpoint | `val_loss improved (... → ...), saving model...` |
| Plateau | `No improvement. EarlyStopping: n/5 | ReduceLR: n/3` |
| ReduceLR | `ReduceLROnPlateau: LR diturunkan → ...` |
| Stop / restore | `EarlyStopping triggered!`, `Training selesai!`, `Best model loaded!` |
| Output | .keras, TensorBoard, history.csv, metrics.csv, metrics_per_class.csv, predictions_test.csv, confusion matrix, kurva training |
| Dataset/split | Gambar dan anotasi YOLO lama; target diubah menjadi kehadiran kelas per gambar |

Model: Input → augmentasi train → Rescaling → 4 blok Conv2D/BatchNorm/ReLU/MaxPool (32/64/128/256) → GlobalAveragePooling → Dense48/ReLU → Dropout0.4 → Dense5/Sigmoid. Bobot awal acak; tidak ada pretrained backbone. Semua kelas yang muncul di gambar dipertahankan.

**Interpretasi evaluasi:** akurasi pada log menghitung lima keputusan biner untuk setiap gambar. Exact-match mengharuskan kelima keputusan sekaligus benar. F1 macro/micro membantu melihat apakah kelas kerusakan benar-benar dikenali. Performa ini tidak langsung sebanding dengan mAP deteksi YOLO, karena task dan unit prediksinya berbeda.

**Perbedaan yang sengaja diperbaiki dari acuan:** loop manual memasukkan `model.losses` agar regularisasi L2 berpengaruh; rata-rata epoch dibobot jumlah gambar agar batch terakhir tidak terlalu berpengaruh. Format log dan parameter optimizer/callback tetap mengikuti BiLSTM. Class weights tidak diterapkan, sesuai loop BiLSTM acuan yang juga tidak menggunakannya.

**Verifikasi pembuatan:** seluruh sel kode lolos pemeriksaan syntax dan tiga uji callback lolos (early stopping/plateau, min_delta/reset/NaN, serta batas LR minimum). Uji terintegrasi menggunakan TensorFlow 2.17.1 CPU, arsitektur default 224×224, dua epoch dan batch 2 pada 15 gambar sintetis terpisah juga lolos: forward/backprop, validasi, log, simpan/load best model, evaluasi, confusion matrix, dan inference. Probabilitas sebelum/sesudah penyimpanan cocok, dan file sumber fixture tidak berubah. Data sintetis tersebut bukan dataset jalan asli dan hasilnya tidak disertakan. Training GPU serta evaluasi dataset RDDC2024-ID asli belum dijalankan; jalankan notebook lokal untuk memperoleh hasil eksperimenmu.

Referensi API: [Adam](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam), [BinaryAccuracy](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/BinaryAccuracy), [BinaryCrossentropy](https://www.tensorflow.org/api_docs/python/tf/keras/losses/BinaryCrossentropy).
